# smolagents

**Domain:** Agentic AI  ·  **recommended addition**  ·  **runnable:** yes

A refresher on **smolagents** — Hugging Face's deliberately tiny agent library (the core is ~a thousand lines). Its signature idea is the **CodeAgent**: the LLM writes its actions as a snippet of **Python code** instead of emitting JSON tool calls, and smolagents runs that code in a sandboxed interpreter. Model-agnostic, Hub-integrated, and small enough to read end-to-end.

All worked examples here run **offline on CPU with no API key** by driving the agent loop with a deterministic scripted "model"; a final cell shows the live-LLM shape gated behind a key check.

## 1. What & Why

Most agent frameworks make the LLM *act* by returning a structured **JSON tool call** (`{"name": "get_price", "arguments": {"item": "apple"}}`). That works, but it's clumsy: to chain two tools you need two round-trips, to loop or branch you can't, and composing results means inventing more JSON. smolagents' bet — borrowed from the **CodeAct** line of research — is that since LLMs are already excellent at writing Python, the most natural action *is* Python:

```python
unit = get_price("apple")     # call a tool
total = unit * 6              # do arithmetic
final_answer(total)          # return
```

One code blob can call several tools, store intermediate variables, loop, and branch — work that would take many JSON round-trips. That means **fewer steps, fewer tokens, and more expressive actions**.

**What smolagents is:** a minimal library offering two agent types (`CodeAgent`, `ToolCallingAgent`), a uniform `Model` wrapper over basically any backend (HF Inference, LiteLLM → 100+ providers, OpenAI, Azure, Bedrock, local Transformers/MLX/vLLM), a `@tool` decorator, sandboxed Python execution, and first-class Hugging Face Hub sharing of tools and agents.

**Reach for it when:** you want a lightweight, hackable agent; you like code-as-action; you're in the HF ecosystem; or you want to prototype an agent in a dozen lines. **Look elsewhere when:** you need heavy stateful orchestration (graphs, checkpoints, human-in-the-loop branching → LangGraph), or you must run untrusted model-written code in production without standing up a real sandbox.

## 2. Mental Model

**The agent is a programmer sitting at a Python REPL, and the LLM is the programmer.**

Each step of the loop:

1. The model reads the task + history and **writes a `<code>` blob** (its action).
2. smolagents **executes that code** in a Python interpreter where your tools are just functions already in scope.
3. Whatever the code `print`s — plus errors — becomes the **observation** fed back to the model.
4. Repeat until the model calls the special `final_answer(...)` function, which ends the loop and returns the value.

```
task ──▶ [LLM writes Python] ──▶ [sandboxed exec] ──▶ stdout/result
              ▲                                            │
              └──────────────── observation ◀─────────────┘
                         (until final_answer is called)
```

This is exactly the **ReAct** loop (Thought → Action → Observation), but the *Action* is executable code instead of a parsed `tool[arg]` string. The two agent flavors differ only in what the action *is*:

- **`CodeAgent`** — action = a Python snippet (the smolagents default and differentiator).
- **`ToolCallingAgent`** — action = a classic JSON tool call (use when your backend's native function-calling is better, or a tool must not be expressible as free code).

## 3. Key Concepts

| Concept | What it is |
|---|---|
| **`CodeAgent`** | Agent whose action each step is a Python code blob, executed in an interpreter. The library's headline feature. |
| **`ToolCallingAgent`** | Agent that emits structured JSON tool calls instead of code — the conventional approach. |
| **`Tool` / `@tool`** | A callable exposed to the agent. The `@tool` decorator turns a typed, docstringed function into a tool; the **docstring + type hints become the schema** the LLM sees. |
| **`Model`** | Uniform wrapper over an LLM backend: `InferenceClientModel` (HF), `LiteLLMModel` (100+ providers), `OpenAIServerModel`, `TransformersModel`, `MLXModel`, `VLLMModel`, … A `Model` is just a callable: messages in, a `ChatMessage` out. |
| **`final_answer(x)`** | The built-in tool that terminates the loop and returns `x`. Without it the agent runs until `max_steps`. |
| **`LocalPythonExecutor`** | The custom interpreter that runs CodeAgent's code. It is **not arbitrary Python**: imports are restricted to an allow-list (`additional_authorized_imports`) and dangerous ops are blocked. |
| **`additional_authorized_imports`** | The allow-list of modules code may import. Empty by default; widen it deliberately (e.g. `["math", "pandas"]`). |
| **`max_steps`** | Hard cap on loop iterations — your guard against infinite loops / runaway cost. |
| **Managed agents** | An agent can be handed *other* agents as if they were tools, enabling simple multi-agent hierarchies. |
| **`ActionStep` / memory** | Each iteration is recorded as a step in `agent.memory`; useful for inspection, replay, and debugging. |

## 4. Setup

```bash
pip install smolagents                 # core
pip install "smolagents[toolkit]"      # + default tools (web search, etc.)
pip install "smolagents[litellm]"      # + LiteLLM backend (OpenAI/Anthropic/...)
pip install "smolagents[transformers]" # + run local HF models
```

Examples 1–3 below need **only the core install** and run offline. Example 4 (live LLM) additionally needs a token in `HF_TOKEN` (or any other backend's key) and is skipped automatically when absent.

In [1]:
# Environment check. Examples 1-3 need only `smolagents`; Example 4 needs a token.
import importlib.util, os

has_smolagents = importlib.util.find_spec("smolagents") is not None
has_hf_token = bool(os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN"))

print("smolagents installed:", has_smolagents)
print("HF token present (for Example 4):", has_hf_token)

if has_smolagents:
    import smolagents
    print("smolagents version:", smolagents.__version__)

smolagents installed: True
HF token present (for Example 4): False
smolagents version: 1.26.0


/Users/danieldekerlegand/Development/ai-tutor/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 5. Worked Examples

Four examples, increasing in scope:

1. **Define a tool** with `@tool` and see the schema the LLM receives.
2. **Run code in the sandbox** with `LocalPythonExecutor` — including watching it block an unsafe import.
3. **Drive a full `CodeAgent` loop offline** using a deterministic scripted model (no API key, fully reproducible).
4. **The live-LLM shape**, gated behind a token check.

### Example 1 — A tool is just a typed, documented function

The `@tool` decorator inspects the signature and docstring to build the JSON schema the model sees. Good type hints and a clear `Args:` block aren't optional polish — they *are* the interface the LLM reads to decide how to call your tool.

In [2]:
from smolagents import tool

@tool
def get_price(item: str) -> float:
    """Return the unit price of a grocery item in USD.

    Args:
        item: the item name, e.g. "apple" or "banana".
    """
    catalog = {"apple": 0.50, "banana": 0.30, "cherry": 2.00}
    return catalog.get(item.lower(), 1.00)

print("name:        ", get_price.name)
print("description: ", get_price.description.strip())
print("inputs:      ", get_price.inputs)
print("output_type: ", get_price.output_type)
print("direct call: ", get_price("Apple"))  # tools are still ordinary callables

name:         get_price
description:  Return the unit price of a grocery item in USD.
inputs:       {'item': {'type': 'string', 'description': 'the item name, e.g. "apple" or "banana".'}}
output_type:  number
direct call:  0.5


### Example 2 — The Python sandbox (the heart of `CodeAgent`)

A `CodeAgent` doesn't `exec()` raw Python — it uses `LocalPythonExecutor`, a restricted interpreter. Imports must be on an allow-list, so model-written code can't trivially `import os` and wreck your machine. Below we run a harmless snippet, then watch the sandbox reject an unauthorized import.

In [3]:
from smolagents import LocalPythonExecutor

executor = LocalPythonExecutor(additional_authorized_imports=["math"])
executor.send_tools({})  # no tools needed for this demo

result = executor("""
import math
radii = [1, 2, 3]
areas = [math.pi * r**2 for r in radii]
sum(areas)
""")
print("authorized run ->", round(result.output, 4))

# Now try something the sandbox forbids:
try:
    executor("import os; os.system('echo pwned')")
except Exception as e:
    print("blocked     ->", type(e).__name__, "-", str(e).split(" due to:")[-1].strip()[:60])

authorized run -> 43.9823
blocked     -> InterpreterError - InterpreterError: Import of os is not allowed. Authorized im


### Example 3 — A full `CodeAgent` loop, offline and reproducible

To exercise the real agent loop without an API key, we subclass `Model` with a **scripted** stand-in that replays canned code blobs — the same trick a deterministic test harness uses. The `CodeAgent` machinery (parse → execute in sandbox → feed observation back → repeat) is 100% real; only the "brain" is faked. Notice the agent calls `get_price` and then `final_answer` exactly as a true LLM would.

In [4]:
from smolagents import CodeAgent
from smolagents.models import Model, ChatMessage, MessageRole

class ScriptedModel(Model):
    """Offline stand-in for an LLM: returns pre-written code blobs in order."""
    def __init__(self, scripts):
        super().__init__()
        self.scripts, self.i = scripts, 0

    def generate(self, messages, stop_sequences=None, response_format=None,
                 tools_to_call_from=None, **kwargs):
        blob = self.scripts[self.i]
        self.i += 1
        return ChatMessage(role=MessageRole.ASSISTANT, content=blob)

scripts = [
    # Step 1: call the tool, compute, print an observation.
    "Thought: I need the unit price, then multiply by 6.\n"
    "<code>\nunit = get_price('apple')\ntotal = unit * 6\nprint('unit price', unit)\n</code>",
    # Step 2: return the result.
    "Thought: I have the total, so I'll return it.\n"
    "<code>\nfinal_answer(total)\n</code>",
]

agent = CodeAgent(tools=[get_price], model=ScriptedModel(scripts), verbosity_level=0)
answer = agent.run("What do 6 apples cost?")
print("final answer ->", answer)
print("steps taken  ->", sum(1 for s in agent.memory.steps if type(s).__name__ == 'ActionStep'))

final answer -> 3.0
steps taken  -> 2


### Example 4 — The live-LLM shape (gated)

In real use you swap the scripted model for a real one. This is the *only* change needed — everything else (tools, sandbox, loop) is identical. The cell self-skips when no token is set, so the notebook still runs top-to-bottom.

In [5]:
# Gated: runs only if a Hugging Face token is available.
if has_hf_token:
    from smolagents import CodeAgent, InferenceClientModel

    model = InferenceClientModel(model_id="Qwen/Qwen2.5-Coder-32B-Instruct")
    agent = CodeAgent(tools=[get_price], model=model, verbosity_level=0)
    print(agent.run("What is the combined cost of 6 apples and 4 bananas?"))
else:
    print("No HF token set - skipping live call. The code above is the full shape:")
    print("  model = InferenceClientModel(model_id=...)")
    print("  agent = CodeAgent(tools=[get_price], model=model)")
    print("  agent.run('...')")

No HF token set - skipping live call. The code above is the full shape:
  model = InferenceClientModel(model_id=...)
  agent = CodeAgent(tools=[get_price], model=model)
  agent.run('...')


## 6. Gotchas & Pitfalls

- **`LocalPythonExecutor` is a guardrail, not a jail.** It blocks unauthorized imports and obviously dangerous ops, but it is *not* a hardened sandbox. For untrusted tasks or production, run code in a real isolated environment: `executor_type="docker"` or `"e2b"` (`E2BExecutor`). Never point a CodeAgent at a powerful, unsandboxed environment and untrusted input.
- **`additional_authorized_imports` is a footgun in both directions.** Too narrow and the agent's code keeps erroring (`pandas is not allowed`); too wide (`subprocess`, `os`) and you've handed the model your shell. Add the minimum the task needs.
- **Model quality dominates.** Code-writing agents are only as good as the model's coding ability. Tiny/weak models produce buggy blobs and loop forever; use a capable instruction- or code-tuned model (Qwen-Coder, GPT-4-class, Claude, etc.).
- **Docstrings and type hints *are* the API.** The `@tool` schema is built from them. A vague docstring or missing `Args:` means the LLM mis-calls the tool. Treat them as prompt engineering.
- **Always set `max_steps`.** Without a sensible cap, a confused agent burns tokens (and money) looping. It defaults to a finite value — tune it down for cheap tasks.
- **`final_answer` is mandatory to stop.** A CodeAgent that never calls `final_answer(...)` runs until `max_steps` and then errors. If results look truncated, check the model is actually terminating.
- **Errors compound across steps.** A code action that throws is fed back as an observation; a good model self-corrects, but a weak one snowballs. Inspect `agent.memory.steps` when debugging.
- **Code tags / parsing.** CodeAgent expects code in the configured blob tags (e.g. `<code>…</code>` in recent versions). A model that wraps code differently won't parse — match the version's prompt template or set `code_block_tags`.

## 7. When to Use vs Alternatives

| Option | Paradigm | Choose it when… | Cost vs smolagents |
|---|---|---|---|
| **smolagents** | Minimal **code-as-action** agents | You want a tiny, hackable agent, code actions, and HF Hub integration | — |
| **ToolCallingAgent** (in smolagents) | JSON tool calls | A tool must not be free-form code, or the backend's native function-calling is stronger | Same library, just `ToolCallingAgent` |
| **LangGraph** | Explicit **stateful graphs** | You need branching workflows, checkpoints, human-in-the-loop, durable state | Much heavier; more control, more ceremony |
| **CrewAI** | **Role-based** multi-agent crews | You want opinionated "agent personas + tasks" orchestration out of the box | Higher-level, less low-level control |
| **OpenAI Agents SDK** | Handoffs + guardrails | You're committed to OpenAI and want their first-party agent loop | Vendor-leaning; smolagents is model-agnostic |
| **Raw ReAct / tool-calling loop** | Hand-rolled prompt loop | You want zero deps and full control of every token | You rebuild sandbox, parsing, memory yourself |

**The honest summary:** smolagents wins on *smallness and the code-action idea*. Its sweet spot is agents that do real computational work (data wrangling, math, multi-tool composition) where one Python blob replaces five JSON round-trips. Its weak spots are heavy orchestration (reach for LangGraph) and running untrusted code without standing up Docker/E2B isolation. It is **not** a workflow engine — it's a tight agent loop with an unusually good action representation. Cross-link: see [`react.ipynb`](react.ipynb) for the underlying loop and [`langgraph.ipynb`](langgraph.ipynb) for the stateful-graph alternative.

## 8. Resources

- **Official docs** — https://huggingface.co/docs/smolagents
- **GitHub repo** (small enough to read) — https://github.com/huggingface/smolagents
- **Launch blog: "Introducing smolagents"** — https://huggingface.co/blog/smolagents
- **Guided tour & "Building good agents"** — https://huggingface.co/docs/smolagents/tutorials/building_good_agents
- **Secure code execution (sandboxing) docs** — https://huggingface.co/docs/smolagents/tutorials/secure_code_execution
- **CodeAct paper — "Executable Code Actions Elicit Better LLM Agents"** (Wang et al., 2024), the research behind code-as-action — https://arxiv.org/abs/2402.01030